load the movielens ml-1m-unzipped file

In [2]:
import io
import urllib.request
import zipfile
import pandas as pd

# Download MovieLens 1M dataset directly into Colab's temporary environment
url = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
print("Downloading dataset...")
with urllib.request.urlopen(url) as response:
  with zipfile.ZipFile(io.BytesIO(response.read())) as z:
    z.extractall("movielens_data")

# Load the ratings file
ratings = pd.read_csv(
    "movielens_data/ml-1m/ratings.dat",
    sep="::",
    names=["user_id", "movie_id", "rating", "timestamp"],
    engine="python",
)

print("Dataset loaded successfully!")
print(ratings.head(3))

Dataset loaded successfully!
   user_id  movie_id  rating  timestamp
0        1      1193       5  978300760
1        1       661       3  978302109
2        1       914       3  978301968


to isolate the cold-start user slice

In [3]:
# Count how many interactions each user has
user_interaction_counts = ratings["user_id"].value_counts()

# Separate warm users (>= 5 interactions) and cold-start users (< 5 interactions)
cold_start_users = user_interaction_counts[
    user_interaction_counts < 5
].index
warm_users = user_interaction_counts[user_interaction_counts >= 5].index

print(f"Total unique users: {len(user_interaction_counts):,}")
print(f"Warm users (>= 5 interactions): {len(warm_users):,}")
print(f"Cold-start users (< 5 interactions): {len(cold_start_users):,}")

Total unique users: 6,040
Warm users (>= 5 interactions): 6,040
Cold-start users (< 5 interactions): 0


To artificially create the cold start user since the file doesnt have the users less than 5 interactions

In [4]:
import numpy as np
import pandas as pd

# 1. Load movies.dat for item metadata (genres)
movies_path = "movielens_data/ml-1m/movies.dat"
movies = pd.read_csv(
    movies_path,
    sep="::",
    names=["movie_id", "title", "genres"],
    engine="python",
    encoding="latin-1",
)

# 2. Simulate Cold-Start Slice (<5 interactions in training)
np.random.seed(42)
all_users = ratings["user_id"].unique()
cold_start_sample = np.random.choice(
    all_users, size=int(len(all_users) * 0.1), replace=False
)

train_rows = []
test_rows = []

for uid, group in ratings.groupby("user_id"):
  if uid in cold_start_sample:
    k = np.random.randint(1, 5)  # 1 to 4 interactions visible
    sampled = group.sample(n=k, random_state=42)
    train_rows.append(sampled)
    test_rows.append(group.drop(sampled.index))
  else:
    # 80/20 standard train/test split for warm users
    sampled = group.sample(frac=0.8, random_state=42)
    train_rows.append(sampled)
    test_rows.append(group.drop(sampled.index))

train_df = pd.concat(train_rows).reset_index(drop=True)
test_df = pd.concat(test_rows).reset_index(drop=True)

print(f"Total Movies: {len(movies):,}")
print(f"Training interactions: {len(train_df):,}")
print(f"Testing interactions: {len(test_df):,}")
print(f"Cold-start users (<5 ratings in train): {len(cold_start_sample):,}")

Total Movies: 3,883
Training interactions: 725,934
Testing interactions: 274,275
Cold-start users (<5 ratings in train): 604


To build the content based similarity matrix

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Preprocess genre strings (e.g., 'Action|Adventure' -> 'Action Adventure')
movies["genre_clean"] = movies["genres"].str.replace("|", " ", regex=False)

# Compute TF-IDF matrix
tfidf = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
tfidf_matrix = tfidf.fit_transform(movies["genre_clean"])

# Compute item-to-item cosine similarity
content_similarity = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Index mapping for fast lookup
movie_idx_to_id = movies["movie_id"].to_dict()
movie_id_to_idx = {v: k for k, v in movie_idx_to_id.items()}

print("Content-based similarity matrix ready shape:", content_similarity.shape)

Content-based similarity matrix ready shape: (3883, 3883)


To install scikit-suprise and train svd model

In [6]:
# Install the surprise library for collaborative filtering
!pip install scikit-surprise

from surprise import Dataset, Reader, SVD

# 1. Format the training data for Surprise
# We use train_df since that is what we named it in Step 1
reader = Reader(rating_scale=(1, 5))
surprise_data = Dataset.load_from_df(
    train_df[['user_id', 'movie_id', 'rating']],
    reader
)

# Build the full training set
trainset = surprise_data.build_full_trainset()

# 2. Initialize and Train the SVD (Matrix Factorization) Model
print("Training the Collaborative Filtering model (SVD)...")
svd_model = SVD(n_factors=100, random_state=42)
svd_model.fit(trainset)

print("SVD Model trained successfully!")

# Quick test prediction for a specific user and movie
sample_user = train_df['user_id'].iloc[0]
sample_movie = train_df['movie_id'].iloc[0]
prediction = svd_model.predict(sample_user, sample_movie)

print(f"Sample Prediction - User {sample_user} for Movie {sample_movie}: Estimated Rating = {prediction.est:.2f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 10.8 MB/s eta 0:00:00
Training the Collaborative Filtering model (SVD)...
SVD Model trained successfully!
Sample Prediction - User 1 for Movie 2797: Estimated Rating = 4.13


Dynamic blending to avoid 50/50 mix

In [7]:
def get_content_based_score(user_id, movie_id, train_data, sim_matrix, id_to_idx):
    """Calculates a predicted rating based on the genre similarity of movies the user has rated."""
    user_history = train_data[train_data['user_id'] == user_id]

    if user_history.empty or movie_id not in id_to_idx:
        return 3.0  # Neutral fallback if no data exists

    target_idx = id_to_idx[movie_id]

    # Compare target movie to all movies the user has rated
    sim_scores = []
    for _, row in user_history.iterrows():
        rated_movie = row['movie_id']
        if rated_movie in id_to_idx:
            rated_idx = id_to_idx[rated_movie]
            similarity = sim_matrix[target_idx, rated_idx]
            sim_scores.append((similarity, row['rating']))

    total_sim = sum(sim for sim, _ in sim_scores)
    if total_sim == 0:
        return 3.0

    # Return weighted average rating based on genre similarity
    return sum(sim * rating for sim, rating in sim_scores) / total_sim

def hybrid_predict(user_id, movie_id):
    """Dynamically blends CF and CB scores based on user history length."""
    # 1. Get Base Scores
    cf_score = svd_model.predict(user_id, movie_id).est
    cb_score = get_content_based_score(user_id, movie_id, train_df, content_similarity, movie_id_to_idx)

    # 2. Dynamic Weighting Logic
    user_history_len = len(train_df[train_df['user_id'] == user_id])

    # Cap the threshold at 15 interactions.
    # If a user has 3 ratings, CF weight = 20%, CB weight = 80%.
    cf_weight = min(user_history_len / 15.0, 1.0)
    cb_weight = 1.0 - cf_weight

    # 3. Blend
    final_score = (cf_weight * cf_score) + (cb_weight * cb_score)
    return final_score, cf_score, cb_score, cf_weight, cb_weight

# Test the dynamic blender on the sample user
test_score, cf, cb, cf_w, cb_w = hybrid_predict(sample_user, sample_movie)
user_count = len(train_df[train_df['user_id'] == sample_user])

print(f"User {sample_user} has {user_count} interactions in training.")
print(f"CF Score (SVD): {cf:.2f} | Weight: {cf_w:.2f}")
print(f"CB Score (Genres): {cb:.2f} | Weight: {cb_w:.2f}")
print(f"🔥 Final Blended Rating: {test_score:.2f}😎")

User 1 has 42 interactions in training.
CF Score (SVD): 4.13 | Weight: 1.00
CB Score (Genres): 4.02 | Weight: 0.00
🔥 Final Blended Rating: 4.13😎


To run the evaluation loop and report Precision@K, Recall@K, and NDCG@K

In [8]:
import numpy as np
import pandas as pd

# Identify cold-start vs warm users from the training set
interaction_counts = train_df.groupby("user_id").size()
eval_cold_users = interaction_counts[interaction_counts < 5].index.tolist()
eval_warm_users = interaction_counts[interaction_counts >= 5].index.tolist()

# Sample 100 users from each slice for fast, representative evaluation
np.random.seed(42)
sample_cold = np.random.choice(
    eval_cold_users, size=min(100, len(eval_cold_users)), replace=False
)
sample_warm = np.random.choice(
    eval_warm_users, size=min(100, len(eval_warm_users)), replace=False
)

# Relevant items: ratings >= 4.0 in test set
test_relevant = (
    test_df[test_df["rating"] >= 4.0].groupby("user_id")["movie_id"].apply(set)
)


def dcg_at_k(r, k):
  """Calculates Discounted Cumulative Gain at rank K."""
  # np.asarray replaces deprecated np.asfarray for NumPy 2.0 compatibility
  r = np.asarray(r, dtype=float)[:k]
  if r.size:
    return np.sum(r / np.log2(np.arange(2, r.size + 2)))
  return 0.0


def ndcg_at_k(r, k):
  """Calculates Normalized Discounted Cumulative Gain at rank K."""
  dcg_max = dcg_at_k(sorted(r, reverse=True), k)
  if not dcg_max:
    return 0.0
  return dcg_at_k(r, k) / dcg_max


def evaluate_slice(user_list, k=10):
  metrics = {
      "SVD": {"p": [], "r": [], "ndcg": []},
      "Content": {"p": [], "r": [], "ndcg": []},
      "Hybrid": {"p": [], "r": [], "ndcg": []},
  }

  all_movies = movies["movie_id"].values

  for uid in user_list:
    actual_positives = test_relevant.get(uid, set())
    if len(actual_positives) == 0:
      continue

    # Candidate pool: all test movies for user + 50 random unrated items
    user_test_movies = test_df[test_df["user_id"] == uid]["movie_id"].tolist()
    user_train_movies = set(
        train_df[train_df["user_id"] == uid]["movie_id"].tolist()
    )

    unrated_candidates = [
        m
        for m in all_movies
        if m not in user_train_movies and m not in user_test_movies
    ]
    sampled_negatives = list(
        np.random.choice(
            unrated_candidates,
            size=min(50, len(unrated_candidates)),
            replace=False,
        )
    )
    candidate_items = list(set(user_test_movies + sampled_negatives))

    # Score candidates across all 3 strategies
    scores_svd = []
    scores_cb = []
    scores_hybrid = []

    for mid in candidate_items:
      hyb_score, cf_s, cb_s, _, _ = hybrid_predict(uid, mid)
      scores_svd.append((mid, cf_s))
      scores_cb.append((mid, cb_s))
      scores_hybrid.append((mid, hyb_score))

    # Evaluate Precision@K, Recall@K, NDCG@K
    for model_name, score_list in [
        ("SVD", scores_svd),
        ("Content", scores_cb),
        ("Hybrid", scores_hybrid),
    ]:
      score_list.sort(key=lambda x: x[1], reverse=True)
      top_k = [item[0] for item in score_list[:k]]

      hits = [1 if item in actual_positives else 0 for item in top_k]

      prec = sum(hits) / k
      rec = sum(hits) / len(actual_positives)
      ndcg = ndcg_at_k(hits, k)

      metrics[model_name]["p"].append(prec)
      metrics[model_name]["r"].append(rec)
      metrics[model_name]["ndcg"].append(ndcg)

  # Aggregate metrics into DataFrame
  results = {}
  for model_name in ["SVD", "Content", "Hybrid"]:
    results[model_name] = {
        "Precision@10": np.mean(metrics[model_name]["p"]),
        "Recall@10": np.mean(metrics[model_name]["r"]),
        "NDCG@10": np.mean(metrics[model_name]["ndcg"]),
    }
  return pd.DataFrame(results).T


# Run evaluation on both slices
print("Evaluating Cold-Start Slice (<5 interactions)...")
cold_results = evaluate_slice(sample_cold, k=10)

print("Evaluating Warm User Slice (>=5 interactions)...")
warm_results = evaluate_slice(sample_warm, k=10)

# Display formatted benchmark tables
print("\n" + "=" * 55)
print("COLD-START USER EVALUATION (<5 interactions)")
print("=" * 55)
display(cold_results.round(4))

print("\n" + "=" * 55)
print("WARM USER EVALUATION (>=5 interactions)")
print("=" * 55)
display(warm_results.round(4))

Evaluating Cold-Start Slice (<5 interactions)...
Evaluating Warm User Slice (>=5 interactions)...

COLD-START USER EVALUATION (<5 interactions)


,Precision@10,Recall@10,NDCG@10
SVD,0.730,0.1568,0.8983
Content,0.467,0.0961,0.7297
Hybrid,0.671,0.1467,0.8633



WARM USER EVALUATION (>=5 interactions)


,Precision@10,Recall@10,NDCG@10
SVD,0.4909,0.3521,0.8079
Content,0.2202,0.1507,0.4447
Hybrid,0.4909,0.3521,0.8079


In [9]:
# Sample 500 interactions from the test set to hunt for prediction errors
import numpy as np
import pandas as pd
train_df = pd.concat(train_rows).reset_index(drop=True)
test_df = pd.concat(test_rows).reset_index(drop=True)
test_sample = test_df.sample(n=500, random_state=42)
failure_records = []

for _, row in test_sample.iterrows():
    uid = row['user_id']
    mid = row['movie_id']
    actual_rating = row['rating']

    # Get the hybrid prediction
    predicted_rating, cf, cb, cf_w, cb_w = hybrid_predict(uid, mid)

    # Calculate absolute error
    error = abs(actual_rating - predicted_rating)

    # Define a "failure" as a prediction that is off by more than 1.5 stars
    if error > 1.5:
        failure_records.append({
            'user_id': uid,
            'movie_id': mid,
            'actual_rating': actual_rating,
            'predicted_rating': predicted_rating,
            'error': error,
            'cf_weight': cf_w
        })

# Create a DataFrame and merge with movie titles to see what went wrong
failures_df = pd.DataFrame(failure_records)
if not failures_df.empty:
    failures_df = failures_df.sort_values(by='error', ascending=False)
    failures_df = failures_df.merge(movies[['movie_id', 'title', 'genres']], on='movie_id')

    print(f"Found {len(failures_df)} significant failure cases.🫡")
    display(failures_df.head(10).round(2))
else:
    print("No major failures found in this sample!")

Found 73 significant failure cases.🫡


,user_id,movie_id,actual_rating,predicted_rating,error,cf_weight,title,genres
0,812,356,5,1.38,3.62,0.13,Forrest Gump (1994),Comedy|Romance|War
1,4315,1707,1,3.96,2.96,0.20,Home Alone 3 (1997),Children's|Comedy
2,452,534,2,4.96,2.96,1.00,Shadowlands (1993),Drama|Romance
3,18,2042,1,3.84,2.84,0.13,D2: The Mighty Ducks (1994),Children's|Comedy
4,3250,19,5,2.22,2.78,1.00,Ace Ventura: When Nature Calls (1995),Comedy
5,2584,110,2,4.52,2.52,1.00,Braveheart (1995),Action|Drama|War
6,4354,2040,1,3.39,2.39,0.20,"Computer Wore Tennis Shoes, The (1970)",Children's|Comedy
7,990,44,5,2.63,2.37,1.00,Mortal Kombat (1995),Action|Adventure
8,1262,2918,5,2.75,2.25,0.20,Ferris Bueller's Day Off (1986),Comedy
9,3674,3608,1,3.22,2.22,1.00,Pee-wee's Big Adventure (1985),Comedy


Bonus (rerankinng the step for diversity and novelty)

In [10]:
import math

# 1. Calculate global movie popularity (interaction count per movie in training)
movie_popularity = train_df['movie_id'].value_counts().to_dict()

def rerank_for_novelty(user_id, candidate_items, top_k=10):
    """Re-ranks candidate items to penalize highly popular movies and boost novelty."""
    scored_items = []

    for mid in candidate_items:
        # Get the original dynamic hybrid prediction
        hyb_score, _, _, _, _ = hybrid_predict(user_id, mid)

        # Look up how many times this movie was rated (default to 1 if totally new)
        popularity = movie_popularity.get(mid, 1)

        # Apply novelty penalty: divide score by log10 of popularity
        # The +10 prevents division by zero and smooths the penalty curve
        penalty_factor = math.log10(popularity + 10)
        novelty_score = hyb_score / penalty_factor

        scored_items.append({
            'movie_id': mid,
            'hybrid_score': hyb_score,
            'novelty_score': novelty_score,
            'popularity_count': popularity
        })

    # Sort by the new novelty_score instead of the raw hybrid_score
    reranked_df = pd.DataFrame(scored_items).sort_values(by='novelty_score', ascending=False)

    # Merge titles and genres for readability and return Top K
    return reranked_df.head(top_k).merge(movies[['movie_id', 'title', 'genres']], on='movie_id')

# 2. Test the Re-ranker on a sample candidate pool (e.g., 100 random movies)
np.random.seed(42)
sample_candidates = np.random.choice(movies['movie_id'].values, size=100, replace=False)

print("RUNNING BONUS: DIVERSITY / NOVELTY RE-RANKING")
print("-" * 60)
diversity_results = rerank_for_novelty(sample_user, sample_candidates)
display(diversity_results.round(3))

RUNNING BONUS: DIVERSITY / NOVELTY RE-RANKING
------------------------------------------------------------


,movie_id,hybrid_score,novelty_score,popularity_count,title,genres
0,729,4.179,3.872,2,"Institute Benjamenta, or This Dream People Cal...",Drama
1,3172,3.800,3.649,1,Ulysses (Ulisse) (1954),Adventure
2,3295,3.765,3.615,1,Raining Stones (1993),Drama
3,2224,3.710,3.563,1,Downhill (1927),Drama
4,3750,3.710,3.563,1,Boricua's Bond (2000),Drama
5,683,3.710,3.563,1,"Eye of Vichy, The (Oeil de Vichy, L') (1993)",Documentary
6,1045,3.710,3.563,1,Love Is All There Is (1996),Comedy|Drama
7,730,3.710,3.563,1,"Low Life, The (1994)",Drama
8,3151,3.507,3.367,1,"Bat Whispers, The (1930)",Crime|Drama|Mystery
9,3232,3.846,3.356,4,Seven Chances (1925),Comedy
